# B1 - Explore the orders with PyMongo (7 min)
**Slides 11, 15, 17.** Same data you explored in mongosh, now from Python - the way a pipeline sees it.

Run the cells top to bottom (Shift+Enter). Expected outputs are written under each cell.
If the first cell says `No module named 'pymongo'`, ask the instructor (setup step in the handout).

In [ ]:
import os
from pymongo import MongoClient

# Inside dev_env the host is "mongo". Jupyter in this image starts through sudo,
# which drops the container's MONGO_URI variable, so we fall back to the same value.
URI = os.getenv("MONGO_URI", "mongodb://mongo:27017/?directConnection=true")
client = MongoClient(URI, serverSelectionTimeoutMS=5000)
coll = client["shop"]["orders"]
print("connected to", URI, "| orders:", coll.count_documents({}))

Expected: `connected to mongodb://mongo:27017/?directConnection=true | orders: 200`

## 1. Load a sample (slide 22 idea, slide 11 types)
`$sample` gives random documents, so your 3 will differ from your neighbour's. Look at how the types differ.

In [ ]:
for d in coll.aggregate([{"$sample": {"size": 3}}]):
    print(d["order_id"], "| total:", repr(d.get("total")), "| created_at:", repr(d.get("created_at")))

Expected: 3 lines. You will see Python types such as `Decimal128('...')`, `float`, `str`, `datetime.datetime(...)`, `int` or `None`.
PyMongo turns BSON types into Python types: `Decimal128` stays `Decimal128`, BSON `Date` becomes `datetime`.

## 2. Find the type conflicts (slide 11)
The collection is tiny, so we load all 200 documents and count the Python type of each field.

In [ ]:
from collections import Counter

docs = list(coll.find())

def type_profile(field):
    return Counter(type(d[field]).__name__ if field in d else "missing" for d in docs)

print("total     :", type_profile("total"))
print("created_at:", type_profile("created_at"))

Expected:
```
total     : Counter({'Decimal128': 115, 'float': 55, 'str': 19, 'missing': 11})
created_at: Counter({'datetime': 162, 'str': 26, 'Int64': 12})
```
(`Int64` is how PyMongo shows a BSON `long` - the epoch-millis values.)
Same numbers as `$type` in mongosh (decimal 115 / double 55 / string 19 / missing 11; date 162 / string 26 / long 12).

**Reflection:** which Silver column type would you choose for `total`, and what do you do with the 19 strings?

## 3. Flatten the customer email across schema v1 / v2 (slide 15)
v1 orders keep `customer.email`; v2 orders moved it to `customer.contact.email`.

In [ ]:
naive = [d["customer"].get("email") for d in docs]          # only knows the v1 location
print("naive  - missing emails:", sum(e is None for e in naive))

def flat_email(d):
    c = d["customer"]
    return c.get("email") or c.get("contact", {}).get("email")   # v1, else v2

flat = [flat_email(d) for d in docs]
print("fixed  - missing emails:", sum(e is None for e in flat))
print("schema_v:", Counter(d["schema_v"] for d in docs))

Expected:
```
naive  - missing emails: 93
fixed  - missing emails: 0
schema_v: Counter({1: 107, 2: 93})
```
The naive version is the "NULL spike" from slide 15: every v2 order loses its email.

**Reflection:** how would you notice this in production if nobody told you about the v2 release?

## 4. Explode the items array (slide 17)
One row per item, keeping `order_id` + array index as the child key.
Orders with an empty `items` array keep one row with `None` (that is what `explode_outer` does on slide 17).

In [ ]:
rows = []
for d in docs:
    if not d["items"]:
        rows.append({"order_id": d["order_id"], "item_idx": None, "sku": None, "qty": None})
    for idx, it in enumerate(d["items"]):
        rows.append({"order_id": d["order_id"], "item_idx": idx, "sku": it["sku"],
                     "qty": int(it["qty"])})          # qty is sometimes "2" (a string)

print("orders:", len(docs), "| item rows:", len(rows))
print("rows from empty orders:", sum(r["item_idx"] is None for r in rows))
print("string qtys fixed     :", sum(isinstance(it["qty"], str) for d in docs for it in d["items"]))
print(rows[:3])

Expected:
```
orders: 200 | item rows: 381
rows from empty orders: 40
string qtys fixed     : 36
```
followed by the first 3 rows (`ORD-10000` has no items, so its row is all `None`).

**Reflection:** if you used a plain "explode" (inner) instead, how many orders would disappear from the item table?